# Published BlackCAT dynamical black-hole systems

This notebook uses the published BlackCAT dynamical sample from CDS/VizieR (Corral-Santana et al. 2016, DOI 10.26093/cds/vizier.35870061), not the project's synthetic candidate extension. Mass values and limit flags are preserved. Published sample is not completeness-corrected.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import plotly.express as px
root=Path.cwd()
if not (root/"data/processed/blackcat_dynamical.csv").exists(): root=Path("../")
d=pd.read_csv(root/"data/processed/blackcat_dynamical.csv")
d[["Name","mass_msun_text","mass_limit_flag","mass_error_lower_msun","mass_error_upper_msun","mass_function_msun","distance_kpc","orbital_period_hours","dynamical_refs"]]

In [ ]:
# Notebook metadata
__author__ = "Astro Science Research"
__date__ = "2026-10-28"
__version__ = "1.0.0"
__kernel__ = "python3"
import sys
print(f"Python {sys.version}")
print(f"Notebook: {__file__ if '__file__' in dir() else 'interactive'}")


In [ ]:
# Mass function f(M)=P K_2^3/(2πG); independent numerical check using SI unit conversions.
G=6.67430e-11; MSUN=1.988409870698051e30
P=d.orbital_period_hours*3600; K=d.companion_rv_amplitude_km_s*1000
d["mass_function_recomputed_msun"]=P*K**3/(2*np.pi*G)/MSUN
d["mass_function_relative_difference"]=((d.mass_function_recomputed_msun-d.mass_function_msun)/d.mass_function_msun).abs()
d[["Name","mass_function_msun","mass_function_recomputed_msun","mass_function_relative_difference"]].dropna().sort_values("mass_function_relative_difference",ascending=False).head(10)

In [ ]:
q=d.dropna(subset=["mass_msun_text","distance_kpc"]).copy()
q["mass_msun_text"]=pd.to_numeric(q.mass_msun_text,errors="coerce")
fig=px.scatter(q,x="distance_kpc",y="mass_msun_text",color="mass_limit_flag",hover_name="Name",hover_data=["mass_function_msun","orbital_period_hours","dynamical_refs"],template="plotly_dark",title="BlackCAT dynamical masses and distances")
fig.update_layout(xaxis_title="Distance (kpc)",yaxis_title="Reported mass M₁ (M☉)")
fig.show()

The period and velocity inputs and the published f(M) may come from distinct adopted analyses for some sources. Discrepancies are diagnostic and must be checked against the per-source cited papers; do not treat the recomputation as a corrected catalog value.
## Interpretation safeguards

Limit flags such as `>`, `<=`, and `>=` indicate censored constraints; they are not symmetric error bars. Missing mass errors remain missing. Masses are literature estimates with source-specific modeling assumptions.